# 🤖 AVANT Wake Word Training
## AmaVanta — A New Teammate

This notebook trains a custom **"AVANT"** wake word model using **openWakeWord**.
- ✅ 100% FREE — no API keys, no accounts
- ✅ Runs on Google Colab's free GPU (T4)
- ✅ Takes ~30-45 minutes
- ✅ Output: `avant.onnx` — place in `AVANT/data/` folder

### Steps:
1. Run **Setup** cell
2. Upload your voice sample when prompted (optional but improves accuracy)
3. Run **Training** cell
4. Download `avant.onnx` at the end

> **Make sure GPU is enabled:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 1 — Install dependencies
# ═══════════════════════════════════════════════════════
print('Installing dependencies...')
!pip install -q openwakeword==0.6.0
!pip install -q onnxruntime
!pip install -q tensorflow
!pip install -q audiomentations
!pip install -q librosa soundfile
!pip install -q torch torchaudio
!git clone -q https://github.com/dscripka/openWakeWord.git /content/openWakeWord
!pip install -q -r /content/openWakeWord/requirements.txt
print('✅ Dependencies installed')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 2 — Configuration
# ═══════════════════════════════════════════════════════
import os

# ── WAKE WORD (do not change this) ──────────────────────
TARGET_WORD = 'AVANT'
TARGET_WORD_PHONETIC = 'ah-VAHNT'   # Pronunciation guide for TTS synthesis

# ── OUTPUT ──────────────────────────────────────────────
OUTPUT_DIR = '/content/avant_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── TRAINING SETTINGS ───────────────────────────────────
N_SYNTHETIC_SAMPLES = 2000   # More = better accuracy, longer training
N_EPOCHS = 100
BATCH_SIZE = 32

# ── OWNER VOICE SAMPLE (optional — upload yours below) ──
# This is Michael's actual voice — makes AVANT more accurate for his accent
OWNER_VOICE_SAMPLE = None  # Will be set if you upload your file

print(f'🎯 Training wake word: "{TARGET_WORD}"')
print(f'📁 Output directory: {OUTPUT_DIR}')
print(f'🔢 Synthetic samples: {N_SYNTHETIC_SAMPLES}')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 3 — Upload YOUR voice sample (OPTIONAL but recommended)
# Upload the michael_sample_1_16k.wav file from AVANT/data/voice_samples/
# ═══════════════════════════════════════════════════════
from google.colab import files
import shutil

print('Upload your voice sample (michael_sample_1_16k.wav)')
print('This helps AVANT recognize YOUR specific accent and voice.')
print('Skip this cell if you want to use purely synthetic training data.')
print()

try:
    uploaded = files.upload()
    if uploaded:
        for fname, data in uploaded.items():
            dest = f'/content/{fname}'
            with open(dest, 'wb') as f:
                f.write(data)
            OWNER_VOICE_SAMPLE = dest
            print(f'✅ Uploaded: {fname} ({len(data)} bytes)')
    else:
        print('No file uploaded — using synthetic data only')
except Exception as e:
    print(f'Upload skipped: {e}')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 4 — Generate synthetic "AVANT" audio samples
# Uses Google TTS + audiomentations to create varied training data
# ═══════════════════════════════════════════════════════
import os, subprocess, random, numpy as np
import soundfile as sf
from pathlib import Path

POSITIVE_DIR = '/content/avant_positive'
NEGATIVE_DIR = '/content/avant_negative'
os.makedirs(POSITIVE_DIR, exist_ok=True)
os.makedirs(NEGATIVE_DIR, exist_ok=True)

# Install gTTS for text-to-speech synthesis
!pip install -q gtts pydub
from gtts import gTTS
from pydub import AudioSegment
import io

# TTS voices / accents to synthesize AVANT across different speakers
LANGS = ['en', 'en', 'en']  # English variations
TLDS = ['com', 'co.uk', 'com.au', 'ca', 'co.in']  # Different accents

# Phrases that contain the wake word in natural context
WAKE_PHRASES = [
    'AVANT',
    'hey AVANT',
    'okay AVANT',
    'AVANT please',
    'AVANT listen',
    'AVANT!',
    'yo AVANT',
    'AVANT hey',
]

print(f'Generating {N_SYNTHETIC_SAMPLES} synthetic "AVANT" samples...')

count = 0
samples_per_phrase = N_SYNTHETIC_SAMPLES // len(WAKE_PHRASES)

for phrase in WAKE_PHRASES:
    for tld in TLDS:
        for i in range(max(1, samples_per_phrase // len(TLDS))):
            try:
                tts = gTTS(text=phrase, lang='en', tld=tld, slow=False)
                mp3_buf = io.BytesIO()
                tts.write_to_fp(mp3_buf)
                mp3_buf.seek(0)

                # Convert to 16kHz WAV
                audio = AudioSegment.from_mp3(mp3_buf)
                audio = audio.set_frame_rate(16000).set_channels(1)

                # Random speed variation ±15%
                speed = random.uniform(0.85, 1.15)
                audio = audio._spawn(audio.raw_data, overrides={
                    'frame_rate': int(audio.frame_rate * speed)
                }).set_frame_rate(16000)

                out_path = f'{POSITIVE_DIR}/avant_{count:04d}.wav'
                audio.export(out_path, format='wav')
                count += 1
            except Exception as e:
                pass  # Skip failed samples

# Add owner's real voice sample (repeated/augmented) if uploaded
if OWNER_VOICE_SAMPLE and os.path.exists(OWNER_VOICE_SAMPLE):
    print(f'\nAdding owner real voice sample: {OWNER_VOICE_SAMPLE}')
    real_audio = AudioSegment.from_wav(OWNER_VOICE_SAMPLE)
    real_audio = real_audio.set_frame_rate(16000).set_channels(1)
    # Add 50 augmented versions
    for i in range(50):
        speed = random.uniform(0.9, 1.1)
        vol = random.uniform(-3, 3)  # dB
        aug = real_audio._spawn(real_audio.raw_data, overrides={
            'frame_rate': int(real_audio.frame_rate * speed)
        }).set_frame_rate(16000) + vol
        aug.export(f'{POSITIVE_DIR}/michael_real_{i:03d}.wav', format='wav')
    count += 50
    print(f'Added 50 augmented real-voice samples')

print(f'\n✅ Generated {count} positive samples ("AVANT")')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 5 — Generate negative samples (not-AVANT audio)
# Critical: the model needs to learn what NOT to trigger on
# ═══════════════════════════════════════════════════════
from gtts import gTTS
from pydub import AudioSegment
import io, random

# Common words/phrases that sound vaguely similar to AVANT
NEGATIVE_PHRASES = [
    'event', 'advent', 'avid', 'avon', 'avant garde',
    'advance', 'advantage', 'avante', 'Ivan', 'Evan',
    'okay', 'hey', 'hello', 'listen up', 'computer',
    'hey there', 'what time is it', 'play music',
    'the weather is nice today', 'good morning',
    'turn off the lights', 'set an alarm',
    'call mom', 'send a text', 'open the app',
]

neg_count = 0
for phrase in NEGATIVE_PHRASES:
    for tld in ['com', 'co.uk', 'com.au']:
        try:
            tts = gTTS(text=phrase, lang='en', tld=tld)
            mp3_buf = io.BytesIO()
            tts.write_to_fp(mp3_buf)
            mp3_buf.seek(0)
            audio = AudioSegment.from_mp3(mp3_buf).set_frame_rate(16000).set_channels(1)
            audio.export(f'{NEGATIVE_DIR}/neg_{neg_count:04d}.wav', format='wav')
            neg_count += 1
        except:
            pass

print(f'✅ Generated {neg_count} negative samples')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 6 — Train the AVANT wake word model
# ═══════════════════════════════════════════════════════
import sys
sys.path.insert(0, '/content/openWakeWord')

import os, glob, numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from openwakeword.model import Model
import openwakeword
import soundfile as sf

# Download audio embedding model
openwakeword.utils.download_models()

print('Extracting audio embeddings from training data...')

# Load the embedding model for feature extraction
embed_model = Model(wakeword_models=[])

def get_embeddings_from_dir(directory, label):
    embeddings = []
    labels = []
    files = glob.glob(os.path.join(directory, '*.wav'))
    for fpath in files:
        try:
            audio, sr = sf.read(fpath, dtype='int16')
            if sr != 16000:
                continue
            # Get embeddings in chunks
            chunk_size = 1280
            for i in range(0, len(audio) - chunk_size, chunk_size):
                chunk = audio[i:i+chunk_size]
                pred = embed_model.predict(chunk)
                # Extract raw features before final classification
                # Use the melspectrogram embeddings
            embeddings.append(embed_model.preprocessors['melspectrogram'].get_embedding(audio))
            labels.append(label)
        except Exception as e:
            pass
    print(f'  Processed {len(embeddings)} files from {directory}')
    return embeddings, labels

# Alternative approach: use openWakeWord's built-in training pipeline
print('Using openWakeWord training pipeline...')

# Run the built-in training script
os.chdir('/content/openWakeWord')

!python -m openwakeword.train \
    --positive_reference_clips {POSITIVE_DIR} \
    --negative_reference_clips {NEGATIVE_DIR} \
    --output_dir {OUTPUT_DIR} \
    --model_name avant \
    --n_epochs {N_EPOCHS} \
    --target_recall 0.8 \
    --target_false_positive_rate 0.1

print(f'\n✅ Training complete!')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 7 — Convert to ONNX and test
# ═══════════════════════════════════════════════════════
import os, glob

# Find the trained model
model_files = glob.glob(f'{OUTPUT_DIR}/**/*.onnx', recursive=True) + \
              glob.glob(f'{OUTPUT_DIR}/**/*.tflite', recursive=True) + \
              glob.glob(f'{OUTPUT_DIR}/*.onnx') + \
              glob.glob(f'{OUTPUT_DIR}/*.tflite')

print('Found model files:')
for f in model_files:
    size = os.path.getsize(f)
    print(f'  {f} ({size/1024:.1f} KB)')

# Copy the best model to a clean name
if model_files:
    best = model_files[0]
    ext = os.path.splitext(best)[1]
    final_path = f'/content/avant{ext}'
    import shutil
    shutil.copy(best, final_path)
    print(f'\n✅ Model ready: {final_path}')
else:
    print('No model found — check training output above for errors')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 8 — Quick test with your voice sample
# ═══════════════════════════════════════════════════════
import os
from openwakeword.model import Model
import numpy as np
import soundfile as sf

model_path = '/content/avant.onnx'
if os.path.exists(model_path):
    test_model = Model(
        wakeword_models=[model_path],
        inference_framework='onnx'
    )

    if OWNER_VOICE_SAMPLE and os.path.exists(OWNER_VOICE_SAMPLE):
        audio, sr = sf.read(OWNER_VOICE_SAMPLE, dtype='int16')
        chunk_size = 1280
        scores = []
        for i in range(0, len(audio) - chunk_size, chunk_size):
            chunk = audio[i:i+chunk_size]
            pred = test_model.predict(chunk)
            for k, v in pred.items():
                scores.append(v)

        max_score = max(scores) if scores else 0
        print(f'Test on your voice sample:')
        print(f'  Max detection score: {max_score:.4f}')
        print(f'  Threshold: 0.5')
        if max_score >= 0.5:
            print(f'  ✅ DETECTED! AVANT heard you correctly.')
        else:
            print(f'  ⚠️  Score below threshold — model may need more training data')
            print(f'     Try increasing N_SYNTHETIC_SAMPLES to 3000 and re-run')
    else:
        print('No voice sample to test with — upload one in Cell 3 next time')
else:
    print('Model file not found — check Cell 6 output')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 9 — DOWNLOAD YOUR MODEL
# After this, place avant.onnx in your AVANT/data/ folder
# ═══════════════════════════════════════════════════════
from google.colab import files
import os

model_path = '/content/avant.onnx'
if os.path.exists(model_path):
    print('⬇️  Downloading avant.onnx...')
    files.download(model_path)
    print()
    print('=' * 50)
    print('DONE! Next steps:')
    print('=' * 50)
    print()
    print('1. Place the downloaded avant.onnx file here:')
    print('   AVANT/data/avant_wakeword.onnx')
    print()
    print('2. In your .env file, confirm:')
    print('   WAKE_WORD_MODEL_PATH=./data/avant_wakeword.onnx')
    print()
    print('3. Run AVANT:')
    print('   python avant.py --enroll   (first time — enroll your voice)')
    print('   python avant.py             (start listening)')
    print()
    print('4. Say "AVANT" — she will wake up!')
else:
    print('Model not found at expected path.')
    print('Check the output of Cell 6 and 7 for errors.')
    # Try to find any model file
    import glob
    found = glob.glob('/content/**/*.onnx', recursive=True)
    if found:
        print(f'Found at: {found[0]}')
        files.download(found[0])